# 02 - Silver : typage, nettoyage, une ligne par vente

 Ici on **type** les colonnes, on applique des **règles de nettoyage comptées** (lignes et mutations avant / après), puis on garde **une ligne par vente**, écrite en Delta.

In [2]:
import sys
sys.path.append("/home/jovyan/work")

import pandas as pd
from pyspark import StorageLevel
from pyspark.sql import Window
from pyspark.sql import functions as F

from outils.source import creer_session, creer_bucket, LAKE

ANNEE = 2025
BRONZE = f"{LAKE}/bronze/dvf_{ANNEE}"
SILVER = f"{LAKE}/silver/ventes_dvf_{ANNEE}"

In [3]:
spark = creer_session("silver")
creer_bucket()
df = spark.read.format("delta").load(BRONZE)

Spark 4.2.0 (Hadoop 3.5.0) + Delta Lake + data lake S3. Le premier démarrage télécharge des jars : plusieurs minutes.
Session prête.
Bucket « dvf » déjà présent.


## 1. Typage

 Les codes (`code_departement`, `code_commune`...) restent en **texte** pour garder leurs zéros.

In [4]:
TYPES = {
    "date_mutation": "date",
    "valeur_fonciere": "double",
    "surface_reelle_bati": "double",
    "nombre_pieces_principales": "int",
    "longitude": "double",
    "latitude": "double",
}
for col, t in TYPES.items():
    df = df.withColumn(col, F.expr(f"try_cast({col} AS {t})"))

## 2. Journal des règles
`appliquer()` filtre, puis note combien de **lignes** et de **mutations** (une mutation = plusieurs lignes) restent.

In [5]:
journal = []

def compter(df):
    r = df.agg(F.count("*"), F.countDistinct("id_mutation")).first()
    return r[0], r[1]

def appliquer(df, regle, condition, avant):
    out = df.filter(condition)
    apres = compter(out)
    journal.append({"règle": regle, "lignes avant": avant[0], "lignes après": apres[0],
                    "mutations avant": avant[1], "mutations après": apres[1]})
    print(f"{regle} : {avant[1] - apres[1]} mutations retirées")
    return out, apres

etat = compter(df)
print("Départ : %d lignes, %d mutations" % etat)

Départ : 3793113 lignes, 1384341 mutations


In [6]:
# R1 : seulement les ventes (pas d'échanges, d'adjudications...)
df, etat = appliquer(df, "R1 nature = Vente", F.col("nature_mutation") == "Vente", etat)

# R2 : prix renseigné et positif
df, etat = appliquer(df, "R2 prix > 0", F.col("valeur_fonciere") > 0, etat)

R1 nature = Vente : 74381 mutations retirées
R2 prix > 0 : 47002 mutations retirées


## 3. Informations par vente
Le prix est répété sur chaque ligne d'une mutation, et un même logement peut l'être aussi (une ligne par nature de culture de la parcelle). Pour juger la vente entière, on calcule par `id_mutation` : le nombre de logements **distincts**, la présence d'autres locaux, la surface du logement et le prix au m².

In [ ]:
w = Window.partitionBy("id_mutation")
est_logement = F.col("type_local").isin("Maison", "Appartement")
est_autre = F.col("type_local").isNotNull() & ~F.col("type_local").isin("Maison", "Appartement", "Dépendance")

df = (df
            .withColumn("nb_logements", F.size(F.collect_set(F.when(est_logement, F.struct(
          "id_parcelle", "type_local", "surface_reelle_bati", "nombre_pieces_principales", "lot1_numero"))).over(w)))
      .withColumn("nb_autres_locaux", F.sum(F.when(est_autre, 1).otherwise(0)).over(w))
      .withColumn("surface_logement", F.max(F.when(est_logement, F.col("surface_reelle_bati"))).over(w))
      .withColumn("prix_m2", F.col("valeur_fonciere") / F.col("surface_logement")))

df = df.persist(StorageLevel.MEMORY_AND_DISK)  

In [10]:
# R3 : pas de local commercial, industriel ou autre que maison, appartement, dépendance
df, etat = appliquer(df, "R3 pas d'autre local", F.col("nb_autres_locaux") == 0, etat)

# R4a : au moins un logement (sinon : terrain, dépendance seule... pas de prix au m²)
df, etat = appliquer(df, "R4a au moins un logement", F.col("nb_logements") >= 1, etat)

# R4b : un seul logement distinct (avec plusieurs, on ne sait pas répartir le prix)
df, etat = appliquer(df, "R4b un seul logement distinct", F.col("nb_logements") == 1, etat)

R3 pas d'autre local : 72332 mutations retirées
R4a au moins un logement : 325138 mutations retirées
R4b un seul logement distinct : 40679 mutations retirées


## 4. Surface minimale
Un  logement  de 1 ou 2 m² n'en est pas un. Le seuil ne se choisit pas au hasard : on observe (a) la distribution des petites surfaces, (b) ce que sont ces biens (type, nombre de pièces, prix), (c) ce que le filtre de prix au m² donne.


In [11]:
# (a) Distribution des petites surfaces
v = df.filter(est_logement)
q = [0.0005, 0.001, 0.005, 0.01, 0.02, 0.05, 0.10, 0.25, 0.5]
for p, val in zip(q, v.approxQuantile("surface_logement", q, 0.0001)):
    print(f"quantile {p:>6} : {val:>8.1f} m²")

quantile 0.0005 :      9.0 m²
quantile  0.001 :     10.0 m²
quantile  0.005 :     15.0 m²
quantile   0.01 :     18.0 m²
quantile   0.02 :     20.0 m²
quantile   0.05 :     27.0 m²
quantile    0.1 :     36.0 m²
quantile   0.25 :     56.0 m²
quantile    0.5 :     80.0 m²


In [12]:
# (b) Ces petits biens : combien, de quel type, quel prix au m² médian, combien de pièces ?
tranches = v.withColumn("tranche",
    F.when(F.col("surface_logement") < 5, "1) < 5")
     .when(F.col("surface_logement") < 9, "2) 5 à 9")
     .when(F.col("surface_logement") < 15, "3) 9 à 15")
     .when(F.col("surface_logement") < 20, "4) 15 à 20")
     .when(F.col("surface_logement") < 30, "5) 20 à 30")
     .otherwise("6) 30 et +"))

(tranches.groupBy("tranche", "type_local")
         .agg(F.count("*").alias("ventes"),
              F.round(F.percentile_approx("prix_m2", 0.5), 0).alias("prix_m2_median"),
              F.round(F.avg("nombre_pieces_principales"), 1).alias("pieces_moy"))
         .orderBy("tranche", "type_local")
         .show(20, truncate=False))

+----------+-----------+------+--------------+----------+
|tranche   |type_local |ventes|prix_m2_median|pieces_moy|
+----------+-----------+------+--------------+----------+
|1) < 5    |Appartement|60    |152500.0      |1.1       |
|1) < 5    |Maison     |26    |53000.0       |1.5       |
|2) 5 à 9  |Appartement|239   |9083.0        |1.0       |
|2) 5 à 9  |Maison     |39    |5000.0        |0.7       |
|3) 9 à 15 |Appartement|2820  |7500.0        |1.0       |
|3) 9 à 15 |Maison     |278   |2750.0        |1.0       |
|4) 15 à 20|Appartement|10238 |4882.0        |1.0       |
|4) 15 à 20|Maison     |723   |2500.0        |1.1       |
|5) 20 à 30|Appartement|38571 |4442.0        |1.3       |
|5) 20 à 30|Maison     |3639  |2783.0        |1.5       |
|6) 30 et +|Appartement|311202|3133.0        |2.9       |
|6) 30 et +|Maison     |548040|2138.0        |4.3       |
+----------+-----------+------+--------------+----------+



In [13]:
# (c) Petites surfaces dont le prix au m² paraît plausible : le filtre de prix (R6) ne les écartera pas
(v.filter(F.col("surface_logement") < 9)
  .filter(F.col("prix_m2").between(500, 30000))
  .select("valeur_fonciere", "surface_logement", "prix_m2", "type_local", "nombre_pieces_principales", "code_departement")
  .orderBy("surface_logement")
  .show(15))

+---------------+----------------+-------+-----------+-------------------------+----------------+
|valeur_fonciere|surface_logement|prix_m2| type_local|nombre_pieces_principales|code_departement|
+---------------+----------------+-------+-----------+-------------------------+----------------+
|         3000.0|             1.0| 3000.0|Appartement|                        0|              34|
|         6000.0|             1.0| 6000.0|     Maison|                        1|              36|
|        24360.0|             1.0|24360.0|Appartement|                        1|              29|
|         1250.0|             1.0| 1250.0|Appartement|                        0|              71|
|         8000.0|             1.0| 8000.0|     Maison|                        0|              08|
|         8000.0|             1.0| 8000.0|     Maison|                        0|              08|
|         5250.0|             1.0| 5250.0|     Maison|                        0|              34|
|        25000.0|   

 A partir de quelle surface les biens ressemblent-ils à de vrais logements (pièces renseignées, prix au m² cohérent avec leur département) 

In [ ]:
SURFACE_MIN = 9
assert SURFACE_MIN is not None, "Fixer SURFACE_MIN avant de continuer."

df, etat = appliquer(df, f"R5 surface >= {SURFACE_MIN} m²", F.col("surface_logement") >= SURFACE_MIN, etat)

R5 surface >= 9 m² : 402 mutations retirées


## 5. Seuils du prix au m²
Les bornes se choisissent d'après la **distribution**

In [16]:
q = [0.0001, 0.001, 0.002, 0.005, 0.01, 0.02, 0.03, 0.05, 0.95, 0.98, 0.99, 0.995, 0.998, 0.999, 0.9995]
for p, val in zip(q, df.filter(est_logement).approxQuantile("prix_m2", q, 0.0001)):
    print(f"{p:>7} : {val:>12,.1f}")

 0.0001 :          0.0
  0.001 :          9.5
  0.002 :         46.9
  0.005 :        145.2
   0.01 :        277.8
   0.02 :        454.3
   0.03 :        575.0
   0.05 :        759.5
   0.95 :      7,894.7
   0.98 :     10,600.0
   0.99 :     12,500.0
  0.995 :     14,948.7
  0.998 :     19,547.1
  0.999 :     25,564.6
 0.9995 :     33,500.0


In [ ]:
# les prix les plus bas et les plus hauts 
cols = ["valeur_fonciere", "surface_logement", "prix_m2", "type_local", "code_departement"]
df.filter(est_logement).orderBy("prix_m2").select(*cols).show(10)
df.filter(est_logement).orderBy(F.desc("prix_m2")).select(*cols).show(10)

+---------------+----------------+--------------------+-----------+----------------+
|valeur_fonciere|surface_logement|             prix_m2| type_local|code_departement|
+---------------+----------------+--------------------+-----------+----------------+
|            1.0|           725.0|0.001379310344827586|     Maison|              59|
|            1.0|           725.0|0.001379310344827586|     Maison|              59|
|           0.15|            80.0|            0.001875|     Maison|              85|
|           0.15|            80.0|            0.001875|     Maison|              85|
|            1.0|           310.0|0.003225806451612...|     Maison|              33|
|           0.15|            46.0|0.003260869565217391|Appartement|              78|
|            1.0|           265.0|0.003773584905660...|     Maison|              24|
|            1.0|           253.0|0.003952569169960474|     Maison|              49|
|            1.0|           240.0|0.004166666666666667|     Maiso

In [18]:
# Combien de ventes chaque couple de bornes écarte-t-il ?
v = df.filter(est_logement)
total = v.count()
for lo, hi in [(300, 30000), (500, 30000), (1000, 30000)]:
    gardees = v.filter(F.col("prix_m2").between(lo, hi)).count()
    print(f"[{lo} ; {hi}] : {total - gardees:,} ventes écartées ({(total - gardees) / total:.1%})".replace(",", " "))

[300 ; 30000] : 10 633 ventes écartées (1.2%)
[500 ; 30000] : 21 779 ventes écartées (2.4%)
[1000 ; 30000] : 80 641 ventes écartées (8.8%)


In [19]:
PRIX_M2_MIN = 500
PRIX_M2_MAX = 30000
assert PRIX_M2_MIN is not None and PRIX_M2_MAX is not None, "Fixer les deux seuils avant de continuer."

df, etat = appliquer(df, f"R6 prix m² entre {PRIX_M2_MIN} et {PRIX_M2_MAX}",
                     F.col("prix_m2").between(PRIX_M2_MIN, PRIX_M2_MAX), etat)

R6 prix m² entre 500 et 30000 : 19604 mutations retirées


## 6. Une ligne par vente
Chaque vente retenue a un seul logement : on garde la ligne de ce logement.

In [20]:
ventes = (df.filter(est_logement)
          .dropDuplicates(["id_mutation"])     # le logement est répété par nature de culture : on n'en garde qu'une ligne
          .select("id_mutation", "date_mutation",
                  F.year("date_mutation").alias("annee"), F.month("date_mutation").alias("mois"),
                  "valeur_fonciere", "type_local",
                  F.col("surface_reelle_bati").alias("surface_bati"),
                  F.col("nombre_pieces_principales").alias("nb_pieces"),
                  "code_departement", "code_commune", "nom_commune",
                  "id_parcelle", "longitude", "latitude", "prix_m2"))

assert ventes.count() == etat[1], "Il doit y avoir une ligne par mutation retenue."
ventes.write.format("delta").mode("overwrite").save(SILVER)
print("silver/ventes :", spark.read.format("delta").load(SILVER).count(), "ventes")

silver/ventes : 804803 ventes


In [22]:

spark.stop()